# Whole-song-gen — Colab GPU 推論

Wang et al., ICLR 2024（[paper](https://arxiv.org/abs/2405.09901) / [repo](https://github.com/ZZWaang/whole-song-gen)）

- **用途**: 対抗比較用 MIDI を Colab GPU で生成（ローカル GPU 不要）
- **入力**: テキスト条件なし（Form 自動 or `pstring` 指定）
- **出力**: `demo/whole-song-gen-*/generation-*.mid`（ピアノ系マルチトラック）

手順: Runtime → Change runtime type → **GPU** → 上から順に実行。

In [ ]:
# 0) GPU 確認
import torch
assert torch.cuda.is_available(), "Runtime → Change runtime type → GPU を選んでから再実行"
print(torch.cuda.get_device_name(0), torch.__version__)

In [ ]:
# 1) Drive マウント（生成 MIDI の退避先）
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2) 依存インストール（Colab 既存 torch は触らない）
!pip -q install "pretty_midi>=0.2.10" "labml>=0.4.168" gdown Pillow

In [ ]:
# 3) リポジトリ clone
import os
REPO = '/content/whole-song-gen'
if not os.path.isdir(os.path.join(REPO, '.git')):
    !git clone --depth 1 https://github.com/ZZWaang/whole-song-gen.git {REPO}
else:
    print('already cloned', REPO)
os.chdir(REPO)
print('cwd', os.getcwd())

## 4) 重みダウンロード

公式 `download_link.txt` の Drive ファイル:
- `pretrained_models/` … VAE 等（file id `1VnCOt0dVrRM96hIARzJRawnICiEk3ai4`）
- `results_default/` … 4段拡散 ckpt（file id `15hFKcyyWUCybVb9KV2pzGVWIpvQONizR`）

初回は時間がかかる。Drive にキャッシュして次回はそこから展開する。

In [ ]:
# 4) 重み取得 → 所定パスへ展開
import glob
import shutil
import zipfile
from pathlib import Path

CACHE = Path('/content/drive/MyDrive/whole_song_gen_ckpt')
CACHE.mkdir(parents=True, exist_ok=True)

DOWNLOADS = {
    'pretrained_models': {
        'id': '1VnCOt0dVrRM96hIARzJRawnICiEk3ai4',
        'dest': Path(REPO) / 'pretrained_models',
        'probe': 'ec2vae_enc_8bar.pt',  # 展開後に存在する想定ファイル名（違っても下で探索する）
    },
    'results_default': {
        'id': '15hFKcyyWUCybVb9KV2pzGVWIpvQONizR',
        'dest': Path(REPO) / 'results_default',
        'probe': 'frm---',
    },
}

def _already_ready(dest: Path, probe: str) -> bool:
    if (dest / probe).exists():
        return True
    # results_default: frm フォルダ or weights_default.pt
    if list(dest.glob('**/weights_default.pt')) or list(dest.glob('**/weights_best.pt')):
        return True
    if list(dest.glob('*.pt')):
        return True
    return False

def _gdown_to_cache(file_id: str, name: str) -> Path:
    import gdown
    # キャッシュに zip/ファイルがあれば再利用
    existing = [p for p in CACHE.glob(f'{name}.*') if p.suffix != '.tmp']
    existing = [p for p in existing if p.stat().st_size > 1_000_000] or existing
    if existing:
        print('cache hit', existing[0], 'bytes', existing[0].stat().st_size)
        return existing[0]
    out = CACHE / f'{name}.zip'
    url = f'https://drive.google.com/uc?id={file_id}'
    print('gdown', url, '->', out)
    gdown.download(url=url, output=str(out), quiet=False, fuzzy=True)
    assert out.exists() and out.stat().st_size > 0, f'download failed for {name}'
    return out

def _extract_into(archive: Path, dest: Path):
    dest.mkdir(parents=True, exist_ok=True)
    tmp = Path('/content/_wsg_extract') / dest.name
    if tmp.exists():
        shutil.rmtree(tmp)
    tmp.mkdir(parents=True)

    if zipfile.is_zipfile(archive):
        with zipfile.ZipFile(archive, 'r') as zf:
            zf.extractall(tmp)
    else:
        # zip でない単体ファイルならそのままコピー
        shutil.copy2(archive, tmp / archive.name)
        print('warning: not a zip, copied as-is', archive)

    # ネストしたトップディレクトリをほどく
    children = [p for p in tmp.iterdir()]
    root = children[0] if len(children) == 1 and children[0].is_dir() else tmp

    # dest 内の既存 download_link.txt 以外を消してからコピー
    for p in dest.iterdir():
        if p.name == 'download_link.txt':
            continue
        if p.is_dir():
            shutil.rmtree(p)
        else:
            p.unlink()

    for p in root.iterdir():
        target = dest / p.name
        if p.is_dir():
            shutil.copytree(p, target, dirs_exist_ok=True)
        else:
            shutil.copy2(p, target)

for name, meta in DOWNLOADS.items():
    dest = meta['dest']
    if _already_ready(dest, meta['probe']):
        print('ready', dest)
        continue
    archive = _gdown_to_cache(meta['id'], name)
    print('extract', archive, '->', dest)
    _extract_into(archive, dest)
    print('done', name, 'pts=', len(list(dest.glob('**/*.pt'))))

# 期待パスの存在確認（推論デフォルト）
need = [
    Path(REPO) / 'results_default/frm---/v-default/chkpts',
    Path(REPO) / 'results_default/ctp-a-b-/v-default/chkpts',
    Path(REPO) / 'results_default/lsh-a-b-/v-default/chkpts',
    Path(REPO) / 'results_default/acc-a-b-/v-default/chkpts',
]
for p in need:
    print(('OK' if p.exists() else 'MISSING'), p)
print('pretrained pts', list(Path(REPO, 'pretrained_models').glob('*.pt'))[:8])

In [ ]:
# 5) 上流バグ回避: model/__init__.py が os を未 import
from pathlib import Path
init_py = Path(REPO) / 'model' / '__init__.py'
txt = init_py.read_text(encoding='utf-8')
if 'import os' not in txt:
    init_py.write_text('import os\n' + txt, encoding='utf-8')
    print('patched import os into model/__init__.py')
else:
    print('model/__init__.py already imports os')

## 6) 生成

- 試験: `NSAMPLE = 2`
- 比較用: `NSAMPLE = 100`（長い。セッション切断に注意。Drive 退避セルを忘れずに）
- Form 指定例: `PSTRING = 'i4A4A4B8b4A4B8o4'`, `KEY = 7`（G major）
- Form 自動: `PSTRING = None`

In [ ]:
# 6) 推論パラメータ
NSAMPLE = 2          # 比較本番は 100
PSTRING = None       # 例: 'i4A4A4B8b4A4B8o4'
KEY = 0              # 0-11（PSTRING 指定時に使用）
DEMO_DIR = 'demo'
OUT_DRIVE = '/content/drive/MyDrive/whole_song_gen_out'  # MIDI 退避先

In [ ]:
# 7) 生成実行
import os
import subprocess
os.chdir(REPO)

cmd = ['python', 'inference_whole_song.py', '--nsample', str(NSAMPLE), '--demo_dir', DEMO_DIR]
if PSTRING:
    cmd += ['--pstring', PSTRING, '--key', str(KEY)]
print('RUN:', ' '.join(cmd))
subprocess.run(cmd, check=True)

In [ ]:
# 8) 生成 MIDI を Drive へ退避＋一覧
import os
import shutil
from pathlib import Path
from datetime import datetime

src = Path(REPO) / DEMO_DIR
mids = sorted(src.glob('**/generation-*.mid'))
print('midi count', len(mids))
for p in mids[-10:]:
    print(p)

stamp = datetime.now().strftime('%m-%d_%H%M%S')
dst = Path(OUT_DRIVE) / f'run_{stamp}'
dst.mkdir(parents=True, exist_ok=True)
for p in mids:
    rel = p.relative_to(src)
    out = dst / rel
    out.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(p, out)
print('copied to', dst)

## メモ

- ローカルに GPU が無い前提。CPU でも動くコードだが拡散4段なので実用は Colab GPU。
- 文プロンプト不可。FAD/MAD・Q1〜O1 比較用のランダム生成向き。
- スタイルはポップ／ピアノ寄り。ギターバンド参照との差は論文注記対象。
- `model/__init__.py` の `import os` 欠落は upstream 側のバグで、セル5でパッチする。
- 重みが巨大なため、Drive キャッシュ（`whole_song_gen_ckpt`）を残しておくと再実行が楽。